# GameTheory-02e-IESDS-Rationalisabilite-Python

> **Navigation** : [Notebooks](../) | [Série GameTheory](README.md) | [<< 2-NormalForm (palier parent)](GameTheory-02-NormalForm-Python.ipynb) | [02d - Épistémique / Connaissance commune](GameTheory-02d-Epistemique-Connaissance-Commune-Python.ipynb) | [04 - Nash](GameTheory-04-NashEquilibrium-Python.ipynb)

> **Série** : GameTheory-02e — accrétion du palier 02 (forme normale) : la **rationalisabilité** et l'**élimination itérée des stratégies strictement dominées (IESDS)** — le concept qui relie la forme normale (02) à l'équilibre de Nash (04)

## Pourquoi ce carnet

La série part de la **forme normale** ([02](GameTheory-02-NormalForm-Python.ipynb)) et de la **topologie des petits jeux** ([03](GameTheory-03-Topology2x2-CSharp.ipynb)), puis arrive directement à l'**équilibre de Nash** ([04](GameTheory-04-NashEquilibrium-CSharp.ipynb)). Elle saute le concept de solution **fondamental qui les relie** : la **rationalisabilité**.

Une stratégie est **rationalisable** si elle est une meilleure réponse à *une* croyance sur l'adversaire — et ce, de manière **itérée** (donc cohérente avec la rationalité de l'adversaire). L'**IESDS** (élimination itérée des stratégies **strictement** dominées) est l'algorithme qui calcule, de façon effective, l'ensemble des stratégies rationalisables.

Ce que ce carnet établit, **nativement** sur nos briques (`game_theory_utils.py`) :

1. l'IESDS, implémentée et démontrée pas-à-pas sur des jeux concrets ;
2. le **théorème d'indépendance à l'ordre** : pour la dominance **stricte**, l'ensemble des stratégies survivantes ne dépend **pas** de l'ordre d'élimination (contrairement à la dominance faible, vue en [02c](GameTheory-02c-Travelers-Dilemma-Python.ipynb)) ;
3. le **lien rationalisable ⊇ Nash** : toute stratégie jouée dans un équilibre de Nash est rationalisable — et un exemple où l'inclusion est **stricte**.

## 1. Dominance stricte et IESDS

### Définition — dominance stricte

Une stratégie pure $s_i$ d'un joueur $i$ est **strictement dominée** par une autre stratégie $s_i'$ de ce même joueur si, **face à TOUTE** stratégie de l'adversaire :

$$
u_i(s_i', s_{-i}) \;>\; u_i(s_i, s_{-i}) \quad \forall\, s_{-i}
$$

L'opposé de la dominance **faible** (vue en 02c) : ici l'inégalité est **stricte partout**, pas « au moins égale partout et strictement plus quelque part ». Cette stricte est exactement ce qui rendra l'élimination **indépendante de l'ordre** (§3).

### IESDS — l'algorithme

On supprime, à répétition, toute stratégie strictement dominée (par une autre stratégie pure du même joueur), en réévaluant la dominance **sur le jeu réduit** à chaque passe. Les stratégies qui **survivent** sont les stratégies **rationalisables**.

### Implémentation

On réutilise `NormalFormGame` de `game_theory_utils.py` et on écrit `iesds()` : à chaque passe, on cherche les stratégies du joueur strictement dominées **sur les stratégies adverses restantes**, on les retire, et on s'arrête quand aucune stratégie n'est plus dominée.

In [1]:
import numpy as np
from game_theory_utils import NormalFormGame, find_pure_nash

print("Imports OK : numpy, game_theory_utils (NormalFormGame, find_pure_nash)")

def dominated_own(game, player, kept_own, kept_opp):
    """Strategies of `player` strictly dominated by ANOTHER pure strategy of the
    same player, evaluated over the opponent's currently-kept pure strategies."""
    dom = []
    for s in kept_own:
        for t in kept_own:
            if t == s:
                continue
            if player == 0:
                better = all(game.A[t, o] > game.A[s, o] for o in kept_opp)
            else:
                better = all(game.B[o, t] > game.B[o, s] for o in kept_opp)
            if better:
                dom.append(s)
                break
    return dom

def iesds(game, player, verbose=True, order=None):
    """Elimination iteree des strategies STRICTEMENT dominees (pures).

    Returns the list of surviving (rationalizable) strategies of `player`.
    `order` (optional) is a priority list of indices: when several strategies
    are dominated at once, the one earliest in `order` is removed first
    (one at a time). Without `order`, all dominated strategies go at once.
    """
    m, n = game.A.shape
    kept_own = list(range(m)) if player == 0 else list(range(n))
    kept_opp = list(range(n)) if player == 0 else list(range(m))
    steps = 0
    while True:
        dom = dominated_own(game, player, kept_own, kept_opp)
        if not dom:
            break
        steps += 1
        if order is not None:
            dom = [min(dom, key=order.index)]
        else:
            dom = sorted(dom)
        if verbose:
            print(f"  etape {steps} : elimination de {dom}")
        for s in dom:
            kept_own.remove(s)
    return kept_own, steps

print("iesds() + dominated_own() definies")

Imports OK : numpy, game_theory_utils (NormalFormGame, find_pure_nash)
iesds() + dominated_own() definies


## 2. IESDS pas-à-pas sur des jeux concrets

### Deux jeux : l'un s'amincit, l'autre non

On vérifie l'algorithme sur **deux** jeux. Le premier possède des stratégies strictement dominées (l'IESDS les retire) ; le second n'en a aucune (l'IESDS ne fait rien) — le contraste montre que l'algorithme ne « fabrique » pas d'élimination.

In [2]:
# Jeu A : la strategie M domine strictement T et B (on le verra etape par etape).
game_A = NormalFormGame(
    A=[[1, 0], [2, 3], [0, 1]],
    B=[[1, 0], [2, 3], [0, 1]],
    row_labels=["T", "M", "B"], col_labels=["L", "R"],
    name="Jeu A (strategies strictement dominees)",
)
print("=== Jeu A ===")
game_A.display()
surv_A, steps_A = iesds(game_A, player=0)
print(f"Strategies de Ligne survivantes : {[game_A.row_labels[i] for i in surv_A]}  ({steps_A} etape(s))")
print()

# Jeu B : aucune dominance stricte -> l'IESDS ne retire rien.
game_B = NormalFormGame(
    A=[[3, 0], [0, 3]],
    B=[[0, 3], [3, 0]],
    row_labels=["T", "B"], col_labels=["L", "R"],
    name="Jeu B (aucune dominance stricte)",
)
print("=== Jeu B ===")
game_B.display()
surv_B, steps_B = iesds(game_B, player=0)
print(f"Strategies de Ligne survivantes : {[game_B.row_labels[i] for i in surv_B]}  ({steps_B} etape(s))")

=== Jeu A ===

Jeu A (strategies strictement dominees)
                     L             R
------------------------------------
       T  ( 1.0, 1.0)  ( 0.0, 0.0)  
       M  ( 2.0, 2.0)  ( 3.0, 3.0)  
       B  ( 0.0, 0.0)  ( 1.0, 1.0)  
  etape 1 : elimination de [0, 2]
Strategies de Ligne survivantes : ['M']  (1 etape(s))

=== Jeu B ===

Jeu B (aucune dominance stricte)
                     L             R
------------------------------------
       T  ( 3.0, 0.0)  ( 0.0, 3.0)  
       B  ( 0.0, 3.0)  ( 3.0, 0.0)  
Strategies de Ligne survivantes : ['T', 'B']  (0 etape(s))


### Lecture du résultat

Sur le **Jeu A**, l'IESDS retire d'un coup les deux stratégies `T` et `B` de la Ligne — toutes deux strictement dominées par `M` (`2>1` face à L **et** `3>0` / `3>1` face à R) — et ne laisse que **`M`**. Sur le **Jeu B** (du type « matching pennies »), aucune stratégie n'est strictement dominée : l'algorithme fait **zéro** élimination et les deux stratégies survivent. Le contraste est le point : l'IESDS n'invente pas d'élimination, elle ne retire que ce qui l'est **objectivement**.

## 3. Indépendance à l'ordre (dominance stricte)

### L'énoncé

> **Théorème.** Dans un jeu fini, l'ensemble des stratégies qui survivent à l'IESDS **ne dépend pas de l'ordre** dans lequel on élimine les stratégies strictement dominées.

C'est la propriété qui fait de l'IESDS un **concept de solution bien défini** (et non un algorithme à l'arbitraire) : quel que soit le chemin d'élimination, on aboutit au même jeu réduit.

### Pourquoi c'est vrai (raison en une phrase)

Si une stratégie $s$ est strictement dominée dans un jeu, elle l'est **aussi** dans n'importe quel sous-jeu : retirer des stratégies de l'adversaire ne fait que **réduire** le domaine sur lequel $s$ doit être dominée, et une dominance qui tenait **partout** continue de tenir. Réciproquement, éliminer d'abord une stratégie dominée ne « crée » jamais de dominance stricte nouvelle. Les deux propriétés ensemble **figent** l'ensemble survivant, indépendamment du chemin.

### Illustration : deux ordres, un même résultat

Sur le Jeu A, `T` et `B` sont dominées **toutes les deux** à la première passe. On peut donc les retirer dans **n'importe quel ordre**. On le mesure : retirer `T` d'abord, ou `B` d'abord, mène au même survivant.

In [3]:
# Deux ordres d'elimination possibles sur le Jeu A (T et B dominees d'emblee) :
print("Ordre 1 : T avant B  (priorite [T, M, B])")
surv_o1, _ = iesds(game_A, player=0, order=[0, 1, 2])
print("  -> survivantes :", [game_A.row_labels[i] for i in surv_o1])
print("Ordre 2 : B avant T  (priorite [B, M, T])")
surv_o2, _ = iesds(game_A, player=0, order=[2, 1, 0])
print("  -> survivantes :", [game_A.row_labels[i] for i in surv_o2])
print()
print("Indépendance à l'ordre :",
      "OUI — même ensemble survivant" if surv_o1 == surv_o2 else "NON (inattendu !)")

Ordre 1 : T avant B  (priorite [T, M, B])
  etape 1 : elimination de [0]
  etape 2 : elimination de [2]
  -> survivantes : ['M']
Ordre 2 : B avant T  (priorite [B, M, T])
  etape 1 : elimination de [2]
  etape 2 : elimination de [0]
  -> survivantes : ['M']

Indépendance à l'ordre : OUI — même ensemble survivant


### Lecture du résultat

Les deux ordres d'élimination convergent vers le **même** survivant (`M`) : la propriété d'indépendance à l'ordre est **mesurée**, pas simplement affirmée. C'est ce qui distingue nettement la dominance **stricte** (ici, indépendante de l'ordre) de la dominance **faible** (vue en 02c, où l'unicité de l'équilibre dépendait de la convention de tie-breaking — c'est-à-dire de l'ordre).

## 4. Le lien rationalisable ⊇ Nash

### L'énoncé

> **Théorème.** Toute stratégie jouée (avec probabilité positive) dans un **équilibre de Nash** est **rationalisable**. Autrement dit :
> $$ \{\text{stratégies d'un équilibre de Nash}\} \;\subseteq\; \{\text{stratégies rationalisables}\} $$

L'inclusion peut être **stricte** : une stratégie peut être rationalisable (meilleure réponse à *une* croyance cohérente) sans jamais être jouée dans *aucun* équilibre. L'IESDS est donc un **crible plus permissif** que l'équilibre de Nash — il retient tout ce qui est « défendable » face à une croyance raisonnable, alors que Nash n'en retient que ce qui est **mutuellement** défendable.

### Un exemple où l'inclusion est stricte

On construit un jeu 2×2 où une stratégie de la Ligne **survit** à l'IESDS (elle est une meilleure réponse à « Colonne joue L ») mais **n'apparaît dans aucun équilibre de Nash pur**.

In [4]:
# Jeu C : T est une meilleure reponse a "Colonne joue L" (2 > 1), donc rationalisable ;
# mais a l'equilibre de Nash, la Ligne joue B.
game_C = NormalFormGame(
    A=[[2, 0], [1, 1]],
    B=[[0, 2], [1, 1]],
    row_labels=["T", "B"], col_labels=["L", "R"],
    name="Jeu C (rationalisable strictement plus large que Nash)",
)
game_C.display()

rat_C  = iesds(game_C, player=0, verbose=False)[0]
ne_C   = find_pure_nash(game_C)
nash_C = sorted({i for i, j in ne_C})

print(f"Strategies rationalisables (survivent a l'IESDS) : {[game_C.row_labels[i] for i in rat_C]}")
print(f"Equilibres de Nash purs                            : {[(game_C.row_labels[i], game_C.col_labels[j]) for i, j in ne_C]}")
print(f"Strategies de Ligne dans un equilibre de Nash pur  : {[game_C.row_labels[i] for i in nash_C]}")
print()
print("T est-elle rationalisable ?", "OUI" if 0 in rat_C else "non")
print("T est-elle dans un equilibre de Nash pur ?", "OUI" if 0 in nash_C else "non")


Jeu C (rationalisable strictement plus large que Nash)
                     L             R
------------------------------------
       T  ( 2.0, 0.0)  ( 0.0, 2.0)  
       B  ( 1.0, 1.0)  ( 1.0, 1.0)  
Strategies rationalisables (survivent a l'IESDS) : ['T', 'B']
Equilibres de Nash purs                            : [('B', 'R')]
Strategies de Ligne dans un equilibre de Nash pur  : ['B']

T est-elle rationalisable ? OUI
T est-elle dans un equilibre de Nash pur ? non


### Lecture du résultat

La Ligne a **deux** stratégies rationalisables (`T` et `B`), mais **une seule** apparaît dans un équilibre de Nash pur (`B`). `T` est bien rationalisable — elle est la meilleure réponse à la croyance « Colonne joue L » (`2 > 1`) — et pourtant elle n'est jouée dans **aucun** équilibre : à l'équilibre, Colonne joue `R`, face à laquelle `B` domine `T` (`1 > 0`). C'est l'inclusion stricte **en action** : rationalisable est un crible plus large que Nash.

> **Symétrie** : le même raisonnement tient pour Colonne et pour tout jeu fini. L'IESDS ne « prédit » pas l'équilibre ; elle **borne** le champ des stratégies qu'un joueur rationnel peut défendre. Nash est un raffinement de la rationalisabilité.

## 5. Ce que le carnet a établi

| Résultat | Statut | Où |
|---|---|---|
| IESDS implémentée + démontrée sur ≥ 2 jeux | fait | §2 (Jeu A aminci, Jeu B intact) |
| Indépendance à l'ordre (dominance stricte) | énoncée + mesurée | §3 (deux ordres → même survivant) |
| Rationalisable ⊇ Nash (inclusion stricte) | énoncée + exemple | §4 (Jeu C : `T` rationalisable, hors Nash) |

Le fil conducteur : **forme normale (02) → rationalisabilité par IESDS (02e) → équilibre de Nash (04)**. La rationalisabilité est le maillon manquant — un crible d'« actionnabilité raisonnable » dont Nash est un raffinement.

## 6. Exercices (3) — à compléter par l'étudiant

Trois exercices **C.1** : ils s'exécutent sans erreur mais laissent le calcul à l'étudiant (stubs `return None`).

### Exercice 1 — dominance stricte à la main

Pour le Jeu A, la stratégie `T` est-elle strictement dominée ? Par laquelle ? Écrivez une fonction `est_strictement_dominee(game, player, s)` qui renvoie `True` si la stratégie d'indice `s` du joueur `player` est strictement dominée (par une autre stratégie pure du même joueur, sur **toutes** les stratégies de l'adversaire), `False` sinon.

In [5]:
def est_strictement_dominee(game, player, s):
    """True si la strategie d'indice `s` du joueur `player` est strictement dominee
    par une autre strategie pure du meme joueur (sur TOUTES les strategies de
    l'adversaire)."""
    # --- TODO etudiant ---
    # Indice : pour chaque autre strategie t du meme joueur, verifier que
    # `game.A[t,o] > game.A[s,o]` pour TOUS les o (player==0), ou
    # `game.B[o,t] > game.B[o,s]` pour TOUS les o (player==1).
    return None  # a remplacer par l'implementation de l'etudiant

print("est_strictement_dominee(game_A, 0, 0)  (T du Jeu A) ->",
      est_strictement_dominee(game_A, 0, 0), "  (None tant que non implemente)")

est_strictement_dominee(game_A, 0, 0)  (T du Jeu A) -> None   (None tant que non implemente)


### Exercice 2 — un troisième jeu

Construisez un jeu 3×3 (matrices `A` et `B`) qui possède **au moins une** stratégie strictement dominée pour la Ligne **et** au moins une pour la Colonne, de sorte que l'IESDS retire des stratégies **des deux côtés**. Vérifiez en appelant `iesds()` pour `player=0` et `player=1`.

In [6]:
def jeu_trois_par_trois():
    """Retourne un NormalFormGame 3x3 avec au moins une strategie strictement
    dominee pour la Ligne ET au moins une pour la Colonne."""
    # --- TODO etudiant ---
    # Indice : une ligne (resp. colonne) dont tous les payoffs sont inferieurs a
    # ceux d'une autre ligne (resp. colonne) est strictement dominee.
    # Ex. de depart a adapter : A = [[1,1,1],[3,3,3],[0,2,0]].
    return None  # a remplacer par l'implementation de l'etudiant

g = jeu_trois_par_trois()
if g is None:
    print("jeu_trois_par_trois() -> None  (None tant que non implemente)")
else:
    print("Ligne   :", iesds(g, 0))
    print("Colonne :", iesds(g, 1))

jeu_trois_par_trois() -> None  (None tant que non implemente)


### Exercice 3 — pourquoi Nash raffine la rationalisabilité

Sur le Jeu C, `T` est rationalisable mais hors de tout équilibre de Nash. Écrivez une fonction `dans_un_nash(game, player, s)` qui renvoie `True` si la stratégie d'indice `s` du joueur `player` apparaît dans **au moins un** équilibre de Nash pur (en réutilisant `find_pure_nash`), `False` sinon. Vérifiez qu'elle rend `False` pour `T` et `True` pour `B` du Jeu C.

In [7]:
def dans_un_nash(game, player, s):
    """True si la strategie d'indice `s` du joueur `player` apparait dans au
    moins un equilibre de Nash PUR."""
    # --- TODO etudiant ---
    # Indice : ne = find_pure_nash(game) renvoie une liste de paires (i, j).
    # Pour player==0, tester si `s` est le premier element de l'une de ces paires ;
    # pour player==1, le second.
    return None  # a remplacer par l'implementation de l'etudiant

print("T du Jeu C dans un Nash pur ? ->", dans_un_nash(game_C, 0, 0),
      "  (attendu : False ; None tant que non implemente)")
print("B du Jeu C dans un Nash pur ? ->", dans_un_nash(game_C, 0, 1),
      "  (attendu : True  ; None tant que non implemente)")

T du Jeu C dans un Nash pur ? -> None   (attendu : False ; None tant que non implemente)
B du Jeu C dans un Nash pur ? -> None   (attendu : True  ; None tant que non implemente)


### À retenir

* **Rationalisabilité** : une stratégie est rationalisable si elle survit à l'**IESDS** — elle est une meilleure réponse à une croyance cohérente avec la rationalité de l'adversaire, itérément.
* **Indépendance à l'ordre** : pour la dominance **stricte**, l'ensemble des survivants à l'IESDS **ne dépend pas de l'ordre** d'élimination — c'est ce qui fait de la rationalisabilité un concept bien défini (contrairement à la dominance faible, 02c).
* **Raffinement** : **Nash raffine la rationalisabilité** — toute stratégie d'un équilibre de Nash est rationalisable, mais l'inverse peut échouer (Jeu C : `T` rationalisable, hors de tout Nash). L'IESDS borne le champ de l'« actionnable raisonnable » ; Nash en sélectionne un sous-ensemble mutuellement cohérent.

> **Place dans la série** : ce carnet comble le trou entre la forme normale ([02](GameTheory-02-NormalForm-Python.ipynb)) et l'équilibre de Nash ([04](GameTheory-04-NashEquilibrium-CSharp.ipynb)) — le maillon « rationalisabilité » qui les relie.
>
> **Poursuites** : sibling **C#** (parité avec le reste de la série) et sibling **Lean** (`lean_game_defs`) si le formalisme de l'IESDS s'y prête — laissées en issues de suivi.
>
> **Palier parent** : [GameTheory-02-NormalForm-Python](GameTheory-02-NormalForm-Python.ipynb) · **Précédent** : [GameTheory-02d-Epistémique](GameTheory-02d-Epistemique-Connaissance-Commune-Python.ipynb) · **Série** : [GameTheory](README.md)